[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/05_%E3%81%84%E3%82%8D%E3%81%84%E3%82%8D%E3%81%AA%E7%A2%BA%E7%8E%87%E5%88%86%E5%B8%83%E3%81%A8%E3%83%99%E3%82%A4%E3%82%BA.ipynb)

> 🟢 **Colab（ブラウザ）で実行できます。** Privateリポジトリは初回にColabでGitHub連携の許可が必要です。そのまま上から順に実行できます。

# 統計2級-05. いろいろな確率分布とベイズの定理

2級では二項・正規以外の分布も出ます。代表的なものと、ベイズの定理を学びます。
- ポアソン分布・幾何分布・一様分布・指数分布
- ベイズの定理

In [ ]:
import numpy as np               # 数値計算
import pandas as pd              # 表データ
import matplotlib.pyplot as plt  # グラフ描画
from scipy import stats          # 統計関数（分布・検定など）
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ
rng = np.random.default_rng(0)   # 乱数生成器（seedで結果を固定）

## 1. ポアソン分布

「めったに起きないことが、一定時間に何回起きるか」の分布。平均 λ で決まる。
例：1時間の問い合わせ件数、ページの誤植数。
$$ P(X=k) = \frac{\lambda^k e^{-\lambda}}{k!}, \quad 平均=分散=\lambda $$

In [ ]:
lam = 3   # 1時間に平均3件
k = np.arange(0, 11)                       # 件数の候補（0〜10）
plt.bar(k, stats.poisson.pmf(k, lam))      # ポアソン分布の確率を棒グラフに
plt.title(f'ポアソン分布 λ={lam}'); plt.xlabel('件数'); plt.show()
print('1時間に0件の確率:', round(stats.poisson.pmf(0, lam), 4))      # ちょうど0件
print('5件以上の確率   :', round(1 - stats.poisson.cdf(4, lam), 4))  # 1−(4件以下)

### 二項分布のポアソン近似
$n$ が大きく $p$ が小さいとき、$B(n,p) \approx Poisson(np)$。

In [ ]:
print('B(1000, 0.003) で k=2:', round(stats.binom.pmf(2, 1000, 0.003), 5))  # 二項分布
print('Poisson(3)    で k=2:', round(stats.poisson.pmf(2, 3), 5))           # ポアソン近似（λ=np=3）

## 2. 幾何分布

「初めて成功するまでの試行回数」の分布。例：当たるまでくじを引く回数。
平均は $1/p$。

In [ ]:
p = 0.2                                    # 1回の成功確率
k = np.arange(1, 21)                       # 初成功までの回数の候補
plt.bar(k, stats.geom.pmf(k, p))           # 幾何分布の確率を棒グラフに
plt.title(f'幾何分布 p={p}（平均 {1/p:.0f}回）'); plt.show()
print('3回目で初成功:', round(stats.geom.pmf(3, p), 4))  # ちょうど3回目で初成功

## 3. 一様分布・指数分布（連続）

- **一様分布**：ある区間で同じ確からしさ（例：0〜1の乱数）
- **指数分布**：次の出来事までの待ち時間（ポアソンと対）。平均 $1/\lambda$

In [ ]:
xs = np.linspace(0, 5, 200)                # 横軸の値
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))  # 1行2列のグラフ
ax[0].plot(xs, stats.uniform.pdf(xs, 0, 2)); ax[0].set_title('一様分布 U(0,2)')   # 左：一様分布
ax[1].plot(xs, stats.expon.pdf(xs, scale=1/1.5)); ax[1].set_title('指数分布 λ=1.5')  # 右：指数分布
plt.show()
# 平均2件/時のとき、次の客まで30分以上空く確率
print('指数分布 P(待ち>0.5h):', round(1 - stats.expon.cdf(0.5, scale=1/2), 4))  # 1−(0.5h以下)

## 4. ベイズの定理

「結果」から「原因の確率」を更新する式。
$$ P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)} $$

有名な例：検査の的中率。ある病気の有病率1%、検査の感度99%・特異度95%。
「陽性だったとき、本当に病気である確率」は？

In [ ]:
prior = 0.01                 # 有病率 P(病気)
sens = 0.99                  # 感度 P(陽性|病気)
spec = 0.95                  # 特異度 P(陰性|健康)
p_pos = sens * prior + (1 - spec) * (1 - prior)   # P(陽性)＝病気で陽性＋健康で誤陽性
posterior = sens * prior / p_pos                  # ベイズの定理 P(病気|陽性)
print(f'P(陽性) = {p_pos:.4f}')
print(f'P(病気 | 陽性) = {posterior:.3f}')
print('→ 陽性でも実際に病気の確率は約', round(posterior*100), '%（直感より低い！）')

> 💡 有病率が低いと、感度が高くても「陽性的中率」は低くなる。
これは検診の解釈で実際に問題になる、ベイズの重要な教訓です。

---
## 🏆 練習問題

**問1.** あるサイトのエラーは平均して1日2件（ポアソン）。
1日に0件の確率と、4件以上の確率を求めよう。

**問2.** 成功率10%のガチャ。初当たりが5回目になる確率（幾何分布）と、平均回数を求めよう。

**問3.** 迷惑メールは全体の40%。『無料』を含むメールのうち迷惑は80%、
通常メールで『無料』を含むのは10%。
『無料』を含むメールが迷惑メールである確率をベイズの定理で求めよう。

In [ ]:
# 問1


In [ ]:
# 問2


In [ ]:
# 問3


> 🔑 **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから 👉 **[05_いろいろな確率分布とベイズ の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/05_%E3%81%84%E3%82%8D%E3%81%84%E3%82%8D%E3%81%AA%E7%A2%BA%E7%8E%87%E5%88%86%E5%B8%83%E3%81%A8%E3%83%99%E3%82%A4%E3%82%BA.md)**